# Tiny word embedding

In [8]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("=== STEP 0: Data ===")
docs = ["I love AI", "I love love ML"]
y = np.array([1, 0])  # labels
print("Docs:", docs)
print("Labels (y):", y.tolist())

print("\n=== STEP 1: Tiny word embedding dictionary (3 dimensions) ===")
emb = {
    "i":    np.array([0.10, 0.00, 0.10], dtype=float),
    "love": np.array([0.80, 0.10, 0.00], dtype=float),
    "ai":   np.array([0.60, 0.70, 0.20], dtype=float),
    "ml":   np.array([0.55, 0.65, 0.15], dtype=float),
}
for w, v in emb.items():
    # print(f"  {w:>4s} -> {np.round(v, 3)}")
    pass

def tokenize(doc):
    print(f"\n[tokenize] Input doc: {doc!r}")
    toks = doc.lower().split()
    print(f"[tokenize] Tokens: {toks}")
    return toks

def doc_vector_avg(doc, emb, dim=3):
    print("\n[doc_vector_avg] -------------------------------------------")
    print(f"[doc_vector_avg] Building vector for: {doc!r}")
    tokens = tokenize(doc)
    collected = []
    for t in tokens:
        vec = emb.get(t, np.zeros(dim))
        # print(f"[doc_vector_avg] Token {t!r} -> vector {np.round(vec, 3)}")
        collected.append(vec)
    if len(collected) == 0:
        print("[doc_vector_avg] No tokens found; returning zeros.")
        return np.zeros(dim)
    M = np.vstack(collected)
    # print(f"[doc_vector_avg] Stacked matrix shape: {M.shape}")
    # print(f"[doc_vector_avg] Rows (one per token):\n{np.round(M, 3)}")
    avg = np.mean(M, axis=0)
    # print(f"[doc_vector_avg] Average across rows -> {np.round(avg, 3)}")
    return avg

print("\n=== STEP 2: Build dense document vectors by averaging word vectors ===")
X_rows = []
for i, d in enumerate(docs):
    vec = doc_vector_avg(d, emb, dim=3)
    X_rows.append(vec)
    # print(f"[main] Doc{i} vector -> {np.round(vec, 3)}")


X = np.vstack(X_rows)
print("\n[main] Final design matrix X shape:", X.shape)
print("[main] X:\n", np.round(X, 3))

print("\n=== STEP 3: Train Logistic Regression on dense vectors ===")
clf = LogisticRegression(max_iter=1000)
print("[fit] Starting fit...")
clf.fit(X, y)
print("[fit] Done.")

print("\n=== STEP 4: Inspect learned parameters and predictions ===")
weights = clf.coef_[0]
bias = clf.intercept_[0]
print(f"[params] Weights per embedding dim: dim0={weights[0]:+.4f}, dim1={weights[1]:+.4f}, dim2={weights[2]:+.4f}")
print(f"[params] Bias (intercept): {bias:+.4f}")

pred = clf.predict(X)
proba = clf.predict_proba(X)[:, 1]
print(f"[predict] Predictions: {pred.tolist()}")
print(f"[predict] Probabilities (positive class): {np.round(proba, 4).tolist()}")
print("[metric] Train accuracy:", accuracy_score(y, pred))

=== STEP 0: Data ===
Docs: ['I love AI', 'I love love ML']
Labels (y): [1, 0]

=== STEP 1: Tiny word embedding dictionary (3 dimensions) ===

=== STEP 2: Build dense document vectors by averaging word vectors ===

[doc_vector_avg] -------------------------------------------
[doc_vector_avg] Building vector for: 'I love AI'

[tokenize] Input doc: 'I love AI'
[tokenize] Tokens: ['i', 'love', 'ai']

[doc_vector_avg] -------------------------------------------
[doc_vector_avg] Building vector for: 'I love love ML'

[tokenize] Input doc: 'I love love ML'
[tokenize] Tokens: ['i', 'love', 'love', 'ml']

[main] Final design matrix X shape: (2, 3)
[main] X:
 [[0.5   0.267 0.1  ]
 [0.562 0.213 0.062]]

=== STEP 3: Train Logistic Regression on dense vectors ===
[fit] Starting fit...
[fit] Done.

=== STEP 4: Inspect learned parameters and predictions ===
[params] Weights per embedding dim: dim0=-0.0312, dim1=+0.0270, dim2=+0.0187
[params] Bias (intercept): +0.0086
[predict] Predictions: [1, 0]
[pr

Dense embeddings → Logistic Regression (what happened)

	1.	Input text
You start with two short documents:
	•	Doc0: “I love AI” (label 1)
	•	Doc1: “I love love ML” (label 0)
	2.	Tiny embedding dictionary

Each word (i, love, ai, ml) is assigned a small dense vector (e.g., 3 numbers). Think of this as the word’s coordinates in a semantic space where similar words sit closer.

	3.	Tokenization
Split each sentence into lowercase tokens:
	•	Doc0 → [“i”, “love”, “ai”]
	•	Doc1 → [“i”, “love”, “love”, “ml”]

	4.	Vector lookup & stacking
Replace each token with its embedding vector and stack them into a small matrix:
	•	For Doc0 you get a 3×3 matrix (3 tokens × 3 dims).
	•	For Doc1 you get a 4×3 matrix (4 tokens × 3 dims).

	5.	Average to make a document vector
Take the mean across rows (i.e., average each column).
	•	Result: one 3-D vector per document.
	•	Intuition: the sentence vector is the “center” of its word vectors.

	6.	Design matrix X
Put both doc vectors together → a 2×3 matrix X (2 docs × 3 features).

	7.	Train Logistic Regression
	•	Fit a linear model on X to predict labels (1 vs 0).
	•	It learns weights per embedding dimension + a bias.
	•	Geometrically: it finds a plane in 3D that separates the two points.
	
	8.	Predictions & probabilities
	•	With only two points, it usually classifies both correctly (train acc = 1.0).
	•	Probabilities may hover near 0.5 if the points are close and regularization is active—still, the sign of the linear score decides the class.

Key idea: embeddings compress meaning into a few dimensions, so even simple linear models can separate classes if the semantic averages differ.

# Given tiny word embeddings (3 dims)

i    → [0.10, 0.00, 0.10]
love → [0.80, 0.10, 0.00]
ai   → [0.60, 0.70, 0.20]
ml   → [0.55, 0.65, 0.15]

Doc0: “I love AI”

Tokens: ["i", "love", "ai"]

1) Vector lookup & stacking (rows = tokens, cols = dims)  → shape 3×3
M0 =
[ 0.10  0.00  0.10 ]   # i
[ 0.80  0.10  0.00 ]   # love
[ 0.60  0.70  0.20 ]   # ai

2) Average to make a document vector (column-wise mean)
	•	dim0: (0.10 + 0.80 + 0.60) / 3 = 1.50 / 3 = 0.50
	•	dim1: (0.00 + 0.10 + 0.70) / 3 = 0.80 / 3 ≈ 0.2667
	•	dim2: (0.10 + 0.00 + 0.20) / 3 = 0.30 / 3 = 0.10

Doc0 vector: [0.50, 0.2667, 0.10] (≈ [0.500, 0.267, 0.100])

Doc1: “I love love ML”

Tokens: ["i", "love", "love", "ml"]
(note: “love” appears twice, so it contributes twice to the average)

1) Vector lookup & stacking → shape 4×3

M1 =
[ 0.10  0.00  0.10 ]   # i
[ 0.80  0.10  0.00 ]   # love (1st)
[ 0.80  0.10  0.00 ]   # love (2nd)
[ 0.55  0.65  0.15 ]   # ml


2) Column-wise mean
	•	dim0: (0.10 + 0.80 + 0.80 + 0.55) / 4 = 2.25 / 4 = 0.5625
	•	dim1: (0.00 + 0.10 + 0.10 + 0.65) / 4 = 0.85 / 4 = 0.2125
	•	dim2: (0.10 + 0.00 + 0.00 + 0.15) / 4 = 0.25 / 4 = 0.0625

Doc1 vector: [0.5625, 0.2125, 0.0625] (≈ [0.562, 0.213, 0.063])

Design matrix X

Stack the two document vectors (rows = documents, cols = embedding dims) → shape 2×3:


X =
[ 0.5000  0.2667  0.1000 ]   # Doc0
[ 0.5625  0.2125  0.0625 ]   # Doc1

# Tiny word embeddings (6 dims)

We’ll keep your first 3 dims exactly the same and add 3 more dims for each word:

i    → [0.10, 0.00, 0.10, 0.20, 0.30, 0.40]

love → [0.80, 0.10, 0.00, 0.10, 0.50, 0.30]

ai   → [0.60, 0.70, 0.20, 0.25, 0.15, 0.35]

ml   → [0.55, 0.65, 0.15, 0.30, 0.20, 0.10]


Doc0: “I love AI”

Tokens: ["i", "love", "ai"]

1) Vector lookup & stacking (rows=tokens, cols=dims) → shape 3×6
M0 =

[ 0.10  0.00  0.10  0.20  0.30  0.40 ]   # i

[ 0.80  0.10  0.00  0.10  0.50  0.30 ]   # love

[ 0.60  0.70  0.20  0.25  0.15  0.35 ]   # ai

2) Column-wise mean → Doc0 vector (6 dims)

dim1: (0.10 + 0.80 + 0.60) / 3 = 0.5000

dim2: (0.00 + 0.10 + 0.70) / 3 = 0.2667

dim3: (0.10 + 0.00 + 0.20) / 3 = 0.1000

dim4: (0.20 + 0.10 + 0.25) / 3 = 0.1833

dim5: (0.30 + 0.50 + 0.15) / 3 = 0.3167

dim6: (0.40 + 0.30 + 0.35) / 3 = 0.3500

Doc0 vector:
[0.5000, 0.2667, 0.1000, 0.1833, 0.3167, 0.3500]

Doc1: “I love love ML”

Tokens: ["i", "love", "love", "ml"]
(“love” appears twice, so it’s counted twice in the mean.)

1) Vector lookup & stacking → shape 4×6

M1 =
[ 0.10  0.00  0.10  0.20  0.30  0.40 ]   # i

[ 0.80  0.10  0.00  0.10  0.50  0.30 ]   # love (1st)

[ 0.80  0.10  0.00  0.10  0.50  0.30 ]   # love (2nd)

[ 0.55  0.65  0.15  0.30  0.20  0.10 ]   # ml

2) Column-wise mean → Doc1 vector (6 dims)

dim1: (0.10 + 0.80 + 0.80 + 0.55) / 4 = 0.5625

dim2: (0.00 + 0.10 + 0.10 + 0.65) / 4 = 0.2125

dim3: (0.10 + 0.00 + 0.00 + 0.15) / 4 = 0.0625

dim4: (0.20 + 0.10 + 0.10 + 0.30) / 4 = 0.1750

dim5: (0.30 + 0.50 + 0.50 + 0.20) / 4 = 0.3750

dim6: (0.40 + 0.30 + 0.30 + 0.10) / 4 = 0.2750

Doc1 vector:
[0.5625, 0.2125, 0.0625, 0.1750, 0.3750, 0.2750]

Design matrix 
X
X (rows=documents, cols=embedding dims) → shape 2×6
X =
[ 0.5000  0.2667  0.1000  0.1833  0.3167  0.3500 ]   # Doc0
[ 0.5625  0.2125  0.0625  0.1750  0.3750  0.2750 ]   # Doc1


# Custom word embedding

How important is it?
	•	LLM / RAG / Search engineer (your target): Medium
Know Word2Vec/FastText basics, but emphasize sentence/transformer embeddings (e.g., SBERT, text-embedding models), vector stores, and retrieval quality. Be ready to say when you’d still train custom W2V/FT.
	•	Classic NLP / IR / Recommender ML roles: Medium–High
Expect questions on Word2Vec vs GloVe, negative sampling, subsampling, window size, OOV handling (FastText), and evaluation.
	•	Data Scientist (general): Medium
Conceptual understanding + ability to run a gensim pipeline and use the vectors downstream.

What you should be able to say (and do)
	1.	Concepts (predictive vs count-based):
	•	Word2Vec (CBOW/Skip-gram): predictive; maximize p(context|word) (or reverse); trained with negative sampling and subsampling of frequent words; hyperparams: vector_size, window, min_count, negative, sg, epochs.
	•	GloVe: count-based; factorizes global co-occurrence; objective ~ match w·ŵ + b + b̂ ≈ log X_ij with weighting function.
	•	FastText: subword n-grams → handles OOV / morphology better than Word2Vec.
	2.	When to train custom embeddings:
	•	Domain jargon (e.g., clinical, legal, finance) not well covered by general models.
	•	Low compute / on-premise / privacy constraints (lightweight vs transformers).
	•	Resource-poor languages or heavy misspellings/variants (favor FastText).
	•	Need interpretable, static word space for lexicons, clustering, or feature engineering.
	3.	When not to bother:
	•	If you need contextual meaning (polysemy) or sentence-level tasks → use Sentence-BERT/transformer embeddings.
	•	If pre-trained sentence embeddings already perform well in retrieval/classification benchmarks for your domain.
	4.	Practical pipeline (you should be able to describe it):
	•	Stream corpus → tokenize/normalize → train Word2Vec/FastText (gensim) → save vectors → build doc vectors (avg or TF-IDF-weighted avg) → downstream model (LogReg/SVM) → evaluate (F1/MRR/Recall@k).
	•	For GloVe: build co-occurrence, train (often with the original C impl or ready weights), load into KeyedVectors.
	5.	Evaluation you can talk about:
	•	Intrinsic: nearest neighbors, analogy sanity checks.
	•	Extrinsic: performance in your downstream task (classification/retrieval).
	•	Coverage/OOV rate, t-SNE/UMAP visualization for quick QC.

30-second interview pitch (use this)

“I usually start with pre-trained sentence embeddings for retrieval/classification. If the domain is specialized (e.g., clinical), I’ll try domain-specific sentence models or, when compute/privacy is tight, train FastText/Word2Vec on our corpus. I tune window, min_count, negative, and subsampling; then evaluate intrinsically and, more importantly, extrinsically on our task (F1/MRR). If OOV/misspellings matter, I prefer FastText. Otherwise, for context-sensitive meaning, I use transformers.”

2-minute deeper explanation (if they probe)
	•	Skip-gram with negative sampling: maximize σ(u_c·v_w) for true (w,c) and minimize for sampled negatives; subsampling reduces high-freq stopwords’ dominance; window controls context breadth; min_count trims noise.
	•	GloVe: learn vectors so dot product aligns with log co-occurrence; weighting caps influence of very frequent pairs.
	•	FastText: represents words as the sum of char n-gram vectors, giving robust OOV behavior.
	•	Modern practice: for RAG/search, sentence embeddings (SBERT/LLM embeddings) typically outperform word-averages; but custom W2V/FT are valuable under compute/privacy constraints or for specialized lexicons.

Quick Q → A you might get
	•	Q: Why FastText over Word2Vec?
A: Subword n-grams → handles OOV/misspellings/morphology; better for noisy/inflected domains.
	•	Q: Word2Vec vs GloVe?
A: Predictive (local windows) vs count-based (global co-occurrence); both yield static word embeddings; pick based on tooling/data—results often similar.
	•	Q: How do you turn word vectors into doc vectors?
A: Average or TF-IDF-weighted average; then train a simple classifier or use cosine similarity.
	•	Q: When would you still train W2V today?
A: Domain-specific corpora, limited compute, on-prem privacy, or need for interpretable static spaces.

#  1) Load pretrained Word2Vec + quick sanity checks

In [9]:
# pip install gensim==4.3.3 scikit-learn numpy

from gensim.models import KeyedVectors

W2V_PATH = r"/Users/sameerkhan/Desktop/sameerkhan/data/nlp/GoogleNews-vectors-negative300.bin.gz"  # change me

# Load pretrained word2vec (binary=True for GoogleNews)
w2v = KeyedVectors.load_word2vec_format(W2V_PATH, binary=True)

# 300-dim vectors
print("Vector size:", w2v.vector_size)

# Basic lookups
print("Has 'king'?", "king" in w2v)
print("king[:10] =", w2v["king"][:10])

# Similar words
print("Most similar to 'king':", w2v.most_similar("king", topn=5))

# Analogies: king - man + woman ≈ queen
print("Analogy:", w2v.most_similar(positive=["king", "woman"], negative=["man"], topn=3))


Vector size: 300
Has 'king'? True
king[:10] = [ 0.12597656  0.02978516  0.00860596  0.13964844 -0.02563477 -0.03613281
  0.11181641 -0.19824219  0.05126953  0.36328125]
Most similar to 'king': [('kings', 0.7138046622276306), ('queen', 0.6510956287384033), ('monarch', 0.6413194537162781), ('crown_prince', 0.6204219460487366), ('prince', 0.6159993410110474)]
Analogy: [('queen', 0.7118192911148071), ('monarch', 0.6189674735069275), ('princess', 0.5902431011199951)]


# 2) Sentence embeddings (average Word2Vec) → Logistic Regression

This shows how to convert raw texts into dense sentence vectors (by averaging pretrained word vectors), then train a simple classifier. Includes a small dry-run print for the first sentence so you can see what’s happening step-by-step.


In [18]:
import numpy as np
from gensim.utils import simple_preprocess
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# --- Tiny toy dataset (replace with yours) ---
texts = [
    "I love this movie",
    "This film is terrible",
    "Absolutely fantastic performance",
    "Extremely boring and dull",
    "Superb acting and direction",
    "I will never watch again"
]
y = np.array([1, 0, 1, 0, 1, 0])  # 1=positive, 0=negative

def sent_vec_avg(text: str, model, dim: int = 300, verbose: bool = False):
    """
    Average of pretrained word vectors for tokens present in the model.
    If no token is in vocab, returns zeros.
    """
    toks = simple_preprocess(text)  # lowercase, basic cleanup
    vecs = []
    if verbose:
        print(f"\n[DRY RUN] Text: {text!r}")
        print(f"Tokens: {toks}")
    for t in toks:
        if t in model:
            v = model[t]
            vecs.append(v)
            if verbose:
                print(f"  token {t!r} -> vector[:5]={np.round(v[:5], 3)}")
        else:
            if verbose:
                print(f"  token {t!r} -> OOV (skipped)")
    if not vecs:
        if verbose:
            print("No tokens in vocab -> returning zeros")
        return np.zeros(dim, dtype=np.float32)
    M = np.vstack(vecs)
    if verbose:
        print(f"Stacked shape: {M.shape} (rows=tokens, cols=dims)")
        print(f"Column-wise mean -> sentence vector (first 10 dims): {np.round(M.mean(axis=0)[:10], 3)}")
    return M.mean(axis=0)

# --- Build feature matrix ---
X = np.vstack([sent_vec_avg(t, w2v, w2v.vector_size) for t in texts])
# print(f"X : {X[0]}")

# --- DRY RUN for the first example ---
_ = sent_vec_avg(texts[1], w2v, w2v.vector_size, verbose=True)

# --- Train a simple classifier ---
clf = LogisticRegression(max_iter=1000)
clf.fit(X, y)
pred = clf.predict(X)

print("\nClassification report (on this toy set):")
print(classification_report(y, pred, digits=3))



[DRY RUN] Text: 'This film is terrible'
Tokens: ['this', 'film', 'is', 'terrible']
  token 'this' -> vector[:5]=[ 0.109  0.141 -0.032  0.166 -0.071]
  token 'film' -> vector[:5]=[-0.004 -0.019 -0.131  0.17   0.105]
  token 'is' -> vector[:5]=[ 0.007 -0.073  0.172  0.023 -0.133]
  token 'terrible' -> vector[:5]=[0.164 0.192 0.092 0.131 0.075]
Stacked shape: (4, 300) (rows=tokens, cols=dims)
Column-wise mean -> sentence vector (first 10 dims): [ 0.069  0.06   0.025  0.122 -0.006  0.115  0.058 -0.061  0.108  0.077]

Classification report (on this toy set):
              precision    recall  f1-score   support

           0      1.000     1.000     1.000         3
           1      1.000     1.000     1.000         3

    accuracy                          1.000         6
   macro avg      1.000     1.000     1.000         6
weighted avg      1.000     1.000     1.000         6



# 3) (Optional) Using pretrained Word2Vec in a Keras Embedding layer
If you’re building a neural model and want to initialize the embedding layer with pretrained word vectors:

In [ ]:
# pip install tensorflow keras
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, Model

# 1) Fit a tokenizer on your training text
max_words = 50000
tokenizer = Tokenizer(num_words=max_words, oov_token="<unk>")
tokenizer.fit_on_texts(texts)

print(tokenizer)

# 2) Build an embedding matrix aligned to tokenizer's word_index
vocab_size = min(max_words, len(tokenizer.word_index) + 1) # ["<unk>", "i", "love", "this", "movie", "is", "terrible", "film"] vocab_size = 9, including PAD
embed_dim = w2v.vector_size # “pretrained” vectors are 300-dimensional
embedding_matrix = np.zeros((vocab_size, embed_dim), dtype=np.float32) #(9, 300)

for word, idx in tokenizer.word_index.items():
    if idx >= vocab_size: 
        continue
    if word in w2v:
        embedding_matrix[idx] = w2v[word]  # copy pretrained vector
    # else keeps zeros; you can also set a random small vector

# 3) Convert texts to padded integer sequences
max_len = 40
X_seq = pad_sequences(tokenizer.texts_to_sequences(texts), maxlen=max_len)

# 4) Define a model with an Embedding layer initialized from pretrained weights
inp = layers.Input(shape=(max_len,))
emb = layers.Embedding(
    input_dim=vocab_size,
    output_dim=embed_dim,
    weights=[embedding_matrix],
    trainable=False,        # start frozen; optionally set True to fine-tune
    mask_zero=True
)(inp)
x = layers.Bidirectional(layers.LSTM(64))(emb)
x = layers.Dense(64, activation="relu")(x)
out = layers.Dense(1, activation="sigmoid")(x)
model = Model(inp, out)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

# model.fit(X_seq, y, epochs=5, batch_size=32, validation_split=0.2)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 40)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 40, 300)   │      6,900 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 40)        │          0 │ input_layer_1[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 128)       │    186,880 │ embedding_1[0][0… │
│ (Bidirectional)     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         65 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 202,101 (789.46 KB)

 Trainable params: 195,201 (762.50 KB)

 Non-trainable params: 6,900 (26.95 KB)

In [1]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# -----------------------------
# 1) Tiny toy corpus
# -----------------------------
texts = [
    "I love this movie",
    "This movie is terrible",
    "I love film"
]
# (labels not needed here)

# -----------------------------
# 2) Fit a Tokenizer (with OOV)
# -----------------------------
max_words = 50_000
tokenizer = Tokenizer(num_words=max_words, oov_token="<unk>")
tokenizer.fit_on_texts(texts)

print("document_count:", tokenizer.document_count)
print("vocab size (unique words seen):", len(tokenizer.word_index))
print("oov_token:", tokenizer.oov_token)
print("oov index:", tokenizer.word_index.get(tokenizer.oov_token))
print("\nword_index (token → id):")
for k,v in list(tokenizer.word_index.items())[:10]:
    print(f"  {k:9s} -> {v}")

# -----------------------------
# 3) Build an embedding matrix
#    (toy 'pretrained' 6-d vectors)
# -----------------------------
toy_vec = {
    "i":        np.array([0.10, 0.00, 0.10, 0.20, 0.30, 0.40], dtype=np.float32),
    "love":     np.array([0.80, 0.10, 0.00, 0.10, 0.50, 0.30], dtype=np.float32),
    "this":     np.array([0.20, 0.10, 0.30, 0.20, 0.10, 0.20], dtype=np.float32),
    "movie":    np.array([0.60, 0.30, 0.20, 0.10, 0.40, 0.10], dtype=np.float32),
    "film":     np.array([0.55, 0.25, 0.15, 0.10, 0.35, 0.15], dtype=np.float32),
    "is":       np.array([0.10, 0.05, 0.05, 0.05, 0.05, 0.05], dtype=np.float32),
    "terrible": np.array([0.00, 0.80, 0.20, 0.10, 0.20, 0.70], dtype=np.float32),
}
class MiniW2V:
    def __init__(self, mapping): 
        self.mapping = mapping
        self.vector_size = next(iter(mapping.values())).shape[0]
    def __contains__(self, k): return k in self.mapping
    def __getitem__(self, k): return self.mapping[k]

w2v = MiniW2V(toy_vec)
embed_dim = w2v.vector_size
vocab_size = min(max_words, len(tokenizer.word_index) + 1)  # +1 for PAD (id=0)

embedding_matrix = np.zeros((vocab_size, embed_dim), dtype=np.float32)

# Fill rows from our "pretrained" toy vectors
for word, idx in tokenizer.word_index.items():
    if idx >= vocab_size: 
        continue
    if word == tokenizer.oov_token:
        continue
    if word in w2v:
        embedding_matrix[idx] = w2v[word]

# Give OOV row (index 1) a non-zero vector (mean of known rows)
oov_idx = tokenizer.word_index.get(tokenizer.oov_token, 1)
known_rows = np.stack(list(toy_vec.values()))
embedding_matrix[oov_idx] = known_rows.mean(axis=0)

print("\nembedding_matrix shape:", embedding_matrix.shape, "(rows=vocab_size, cols=embed_dim)")
print("Row 0 (PAD):", embedding_matrix[0])
print("Row 1 (OOV):", np.round(embedding_matrix[1], 3))

# Peek a few known rows
for w in ["i","love","this","movie","film","is","terrible"]:
    print(f"Row[{w:8s} @ {tokenizer.word_index.get(w)}]:", 
          np.round(embedding_matrix[tokenizer.word_index.get(w)], 3))

# -----------------------------
# 4) Convert texts → sequences → pad
# -----------------------------
max_len = 6
seqs = tokenizer.texts_to_sequences(texts)
X_seq = pad_sequences(seqs, maxlen=max_len)  # left-pad with zeros

print("\ntexts_to_sequences:")
for t, s in zip(texts, seqs):
    print(f"  {t!r} -> {s}")

print("\nX_seq shape:", X_seq.shape, "(rows=docs, cols=max_len)")
for i, row in enumerate(X_seq):
    # map back to tokens for clarity (0 -> [PAD])
    tokens = [(tokenizer.index_word.get(x, "[PAD]" if x==0 else "<unk>")) for x in row]
    print(f"Doc{i} ids: {row.tolist()}  ->  tokens: {tokens}")

# -----------------------------
# 5) What the Embedding layer would look up (Doc0)
# -----------------------------
doc0_ids = X_seq[0]
doc0_embedded = embedding_matrix[doc0_ids]   # gather rows by ids
print("\nDoc0 embedded matrix (time_steps × embed_dim):")
print(np.round(doc0_embedded, 3))


document_count: 3
vocab size (unique words seen): 8
oov_token: <unk>
oov index: 1

word_index (token → id):
  <unk>     -> 1
  i         -> 2
  love      -> 3
  this      -> 4
  movie     -> 5
  is        -> 6
  terrible  -> 7
  film      -> 8

embedding_matrix shape: (9, 6) (rows=vocab_size, cols=embed_dim)
Row 0 (PAD): [0. 0. 0. 0. 0. 0.]
Row 1 (OOV): [0.336 0.229 0.143 0.121 0.271 0.271]
Row[i        @ 2]: [0.1 0.  0.1 0.2 0.3 0.4]
Row[love     @ 3]: [0.8 0.1 0.  0.1 0.5 0.3]
Row[this     @ 4]: [0.2 0.1 0.3 0.2 0.1 0.2]
Row[movie    @ 5]: [0.6 0.3 0.2 0.1 0.4 0.1]
Row[film     @ 8]: [0.55 0.25 0.15 0.1  0.35 0.15]
Row[is       @ 6]: [0.1  0.05 0.05 0.05 0.05 0.05]
Row[terrible @ 7]: [0.  0.8 0.2 0.1 0.2 0.7]

texts_to_sequences:
  'I love this movie' -> [2, 3, 4, 5]
  'This movie is terrible' -> [4, 5, 6, 7]
  'I love film' -> [2, 3, 8]

X_seq shape: (3, 6) (rows=docs, cols=max_len)
Doc0 ids: [0, 0, 2, 3, 4, 5]  ->  tokens: ['[PAD]', '[PAD]', 'i', 'love', 'this', 'movie']
Doc1 ids: 

In [2]:
print("\nX_seq shape:", X_seq.shape, "(rows=docs, cols=max_len)")
for i, row in enumerate(X_seq):
    # map back to tokens for clarity (0 -> [PAD])
    tokens = [(tokenizer.index_word.get(x, "[PAD]" if x==0 else "<unk>")) for x in row]
    print(f"Doc{i} ids: {row.tolist()}  ->  tokens: {tokens}")


X_seq shape: (3, 6) (rows=docs, cols=max_len)
Doc0 ids: [0, 0, 2, 3, 4, 5]  ->  tokens: ['[PAD]', '[PAD]', 'i', 'love', 'this', 'movie']
Doc1 ids: [0, 0, 4, 5, 6, 7]  ->  tokens: ['[PAD]', '[PAD]', 'this', 'movie', 'is', 'terrible']
Doc2 ids: [0, 0, 0, 2, 3, 8]  ->  tokens: ['[PAD]', '[PAD]', '[PAD]', 'i', 'love', 'film']
